# App reviews — quick exploration

Questions answered using **`app_reviews_demo.csv`** and the pandas patterns from Week 5.

**Tip:** If `read_csv` fails, set your terminal / notebook working directory to this folder (`Week 5 Demo files`), or adjust the path in the load cell.

In [1]:
import pandas as pd
from pathlib import Path

# Load CSV from this folder or common parent paths (Cursor cwd varies).
_candidates = [
    Path("app_reviews_demo.csv"),
    Path("Week 5 Demo files") / "app_reviews_demo.csv",
    Path("HCDE 530 week 5") / "Week 5 Demo files" / "app_reviews_demo.csv",
]
_path = next((p for p in _candidates if p.exists()), None)
if _path is None:
    raise FileNotFoundError(
        "Could not find app_reviews_demo.csv. Open/run from Week 5 Demo files, or edit _candidates."
    )
df = pd.read_csv(_path)
print(f"Loaded: {_path.resolve()}  rows={len(df)}, cols={len(df.columns)}")

Loaded: /Users/melaniesong/Documents/Code/HCDE 530/HCDE 530 week 5/Week 5 Demo files/app_reviews_demo.csv  rows=500, cols=10


---
## 1. What does your dataset look like?

Use **`df.head()`** to see example rows and **`df.info()`** for column names, dtypes, and non-null counts.

In [2]:
# First rows — shows column values and whether text fields look reasonable.
df.head()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [3]:
# Column list, dtypes, and how many non-null values per column (memory estimate at bottom).
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 500 non-null    int64 
 1   app                500 non-null    object
 2   category           500 non-null    object
 3   rating             500 non-null    int64 
 4   review             500 non-null    object
 5   date               500 non-null    object
 6   helpful_votes      500 non-null    int64 
 7   verified_purchase  500 non-null    bool  
 8   device_type        437 non-null    object
 9   app_version        389 non-null    object
dtypes: bool(1), int64(3), object(6)
memory usage: 35.8+ KB


**Answer (after you run the cells):** Briefly describe what you see — how many rows/columns, what each column represents.

In the dataset, I see 10 columns. There are 500 rows for most columns, with 2 columns device_type and app_version < 500 rows.  

---
## 2. What is the distribution of your most important column?

For review data, **`rating`** is central (1–5). Use **`df["col"].value_counts()`** to count how often each rating appears.

In [4]:
# Count each rating; high counts at 4–5 usually mean skew toward positive reviews.
df["rating"].value_counts().sort_index()

rating
1     29
2     43
3     61
4    160
5    207
Name: count, dtype: int64

In [5]:
# Same counts as proportions (sums to 1.0); easier to describe as "X% are 5 stars".
df["rating"].value_counts(normalize=True).sort_index().mul(100).round(1)

rating
1     5.8
2     8.6
3    12.2
4    32.0
5    41.4
Name: proportion, dtype: float64

**Answer:** State which rating is most common and whether the distribution looks **balanced** or **skewed** (e.g. toward 4–5).
Most common: rating = 5
It seems like the distribution seems to be more skewed towards higher ratings of 4 and 5

---
## 3. Filter to a meaningful subset — what is in it?

Example: reviews with **`rating` < 3** (critical feedback). Pattern: **`df[df["rating"] < 3]`**.

In [6]:
# Boolean mask keeps only rows where the condition is True — here, low ratings.
low_rating = df[df["rating"] < 3]
print(f"Subset size: {len(low_rating)} rows (out of {len(df)} total)")
low_rating.head(10)

Subset size: 72 rows (out of 500 total)


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
26,27,Fieldkit,field research,2,No built-in way to generate a structured debri...,2024-04-04,8,True,mobile,2.5.0
31,32,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-06-27,10,True,tablet,4.2.3
32,33,Lookback,user research,2,Loading large projects takes noticeably longer...,2024-11-22,15,True,desktop,5.2.0
42,43,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-19,9,True,mobile,NaN
51,52,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-09-12,32,True,mobile,NaN
74,75,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...,2023-12-25,14,True,desktop,NaN
79,80,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...,2024-08-02,2,True,desktop,1.8.4
102,103,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...,2024-09-10,16,True,desktop,NaN


In [7]:
# Optional: which apps show up most often among low ratings?
low_rating["app"].value_counts()

app
Fieldkit    22
Miro        17
Lookback    15
Maze        11
Dovetail     7
Name: count, dtype: int64

**Answer:** How many rows are in the subset, what defines it, and one sentence on what the **`head()`** / **`value_counts()`** suggest (e.g. which apps dominate low scores).
There are 72 rows in the subset, out of 500 rows total. It is defined as all reviews with a rating strictly below 3 (one- and two-star reviews only). Counting by app, Fieldkit appears most often in this low-rating slice (22 reviews), followed by Miro (17) and Lookback (15); Dovetail appears least (7). So low scores are not evenly distributed across apps in this demo file—Fieldkit accounts for a larger share of the critical reviews than Dovetail does.

---
## 4. Group by a category and average a numeric column

Pattern: **`df.groupby("app")["rating"].mean()`** — average **`rating`** per **`app`**.

In [ ]:
# One mean rating per app; sort to see highest vs lowest averages at a glance.
df.groupby("app")["rating"].mean().sort_values(ascending=False).round(2)

app
Dovetail    4.12
Miro        4.02
Maze        4.00
Lookback    3.90
Fieldkit    3.67
Name: rating, dtype: float64

In [9]:
summary = (
    df.groupby("app", as_index=False)
    .agg(
        n_reviews=("review", "count"),   # non-null review text
        n_ratings=("rating", "count"),   # non-null rating
    )
    .sort_values("n_reviews", ascending=False)
)
summary

,app,n_reviews,n_ratings
4,Miro,121,121
2,Lookback,105,105
3,Maze,93,93
1,Fieldkit,92,92
0,Dovetail,89,89


**Answer:** Which app has the highest average rating in this sample, which the lowest, and a caution that sample size per app matters (optional: add `.count()` in another line).
The app with highest average rating is Dovetail. However, this could be skewed because there are only 89 reviews in total for this app, where as Miro has 121 reviews. 

---
## 5. Where are the missing values? Are any columns incomplete?

Use **`df.isnull().sum()`** — missing count per column.

In [10]:
# Per-column NaN counts; 0 means complete, larger numbers mean more gaps.
df.isnull().sum()

id                     0
app                    0
category               0
rating                 0
review                 0
date                   0
helpful_votes          0
verified_purchase      0
device_type           63
app_version          111
dtype: int64

In [11]:
# Same as share of rows missing (helps compare columns with different semantics).
df.isnull().mean().mul(100).round(1)

id                    0.0
app                   0.0
category              0.0
rating                0.0
review                0.0
date                  0.0
helpful_votes         0.0
verified_purchase     0.0
device_type          12.6
app_version          22.2
dtype: float64

**Answer:** List which columns have missing values, how many (or what %).

The only 2 columns that have missing data are device_type and app_version. 12.6% is missing for device_type, and 22.2% for app_version. 